# 1. GATED FUSION vs CONCATENATION vs CONDITION D vs EACH MODALITY — one shared protocol — AeroSync-Mamba v29

**Question.** Does content-adaptive gated fusion of new, small modality-specific encoders beat each of the following?
- the current fused model (Condition D);
- plain concatenation of the same new encoders;
- each modality alone.

The comparison uses **one shared protocol**: the same 95 missions, the same paired windows, the same 5 day-out folds,
the same inner validation, the same optimizer and the same seeds for every model.

| model | encoders | fusion | loss |
|---|---|---|---|
| **A** | current audio projection (80 → 256 MLP) | — audio only | class-weighted CE |
| **I** | current IMU projection (2-layer 1D-CNN) | — IMU only | class-weighted CE |
| **D** | current encoders (v22/v24, unchanged) | HTT time OFF; LACA cross-attention with lag bias × 0; `[H_a ; SEP ; H_s]` | CE + 0.1 InfoNCE + 0.1 KL |
| **F0** | NEW (audio feature gate; IMU pre-Conv1d) | plain concatenation `[A ; I]`, no gate, no cross-attention | CE + 0.1 InfoNCE + 0.1 KL |
| **F1** (primary) | NEW | softmax content gate `[g_a, g_i]` on the modality token blocks | CE + 0.1 InfoNCE + 0.1 KL |
| **F2** (secondary, `RUN_F2`) | NEW | two independent sigmoid gates (no sum-to-one) | CE + 0.1 InfoNCE + 0.1 KL |

> ### ⚠️ D vs F0 is NOT a pure "new encoders only" comparison
> Going from D to F0 changes **two things at once**:
> 1. the modality encoders;
> 2. the fusion step. LACA's content-only cross-attention is removed entirely, with no inter-token mixing before
>    Mamba in its place. D's SEP token also goes.
>
> This is an **accepted simplification given time constraints, not an oversight**. F0 does **not** isolate the encoder
> change. The clean comparison in this notebook is **F1 vs F0**: same encoders, the content gate alone.

**Condition D, confirmed from the source.** v22, v24 and v27 have byte-identical `AeroSyncMamba`, `PairwiseLACA`,
`BiModalLACAFusion`, `HeterogeneousTemporalTokenization`, `AeroSyncLoss` and `UAVWindowDataset`.
- **HTT:** the time embedding is OFF, so each token is `LayerNorm(z + e_modality)`.
- **LACA:** it is **active**. It uses real Q/K/V projections, 4 heads and softmax cross-attention in both directions
  (audio→IMU and IMU→audio), with learnable residual scales γ = 0.5 at initialization. The lag-bias term is computed
  but multiplied by β = 0.0, so no timestamp bias is added. **D is not plain concatenation.**
- **Losses:** InfoNCE (τ = 0.07, symmetric, in-batch negatives) and KL consistency are computed from the **mean-pooled
  post-LACA** per-modality token sequences.

**Changes to D that the shared protocol forces** (its architecture is otherwise unchanged):
- **Audio input:** D's audio input is now the shared 80 features (5 de-interleaved channels × 16 bins) instead of v24's
  64 mono strongest bins. So its first Linear is 80 → 256.
- **Audio scaling:** audio is z-scored with the fold-train statistics, as in the audio-only notebook. v24 did not
  normalize audio.

**Consequence:** D's numbers here are not v27's 0.819.

> ### ⚠️ This notebook's table supersedes earlier cross-notebook comparisons
> Earlier numbers came from different window constructions and are **not** directly comparable to each other:
> - D day-out 0.819 (v27);
> - standalone audio 0.845;
> - standalone IMU 0.958.
>
> Only the numbers in section 7 of this notebook are directly comparable to each other.

### Design notes (read before interpreting the results)
- **Paired windows.** These are v24's CORRECTED_ALIGNMENT windows: 2.56 s windows with a 1.28 s stride, starting at
  the audio onset and ending at min(IMU end, audio end). Each window carries 64 audio tokens (the chunks inside it,
  picked evenly by index) and 128 IMU samples (file order, picked evenly by index), **from the same real time span**.
  - The audio notebook's chunk-count windows (128 chunks, stride 64, pair-averaged) are **not** used, because they
    cannot be paired in time with the IMU. The standalone audio number therefore differs from A here.
  - The per-chunk audio feature is exactly the audio notebook's (and v28's A1 feature): sqrt|rFFT| of bins 1–16 of
    the 5 mean-removed channels, with no per-chunk normalization.
- **IMU sampling.** v24 resamples with `np.interp` on the file-order time axis. That is undefined wherever timestamps
  go backwards. Here the 128 real samples are taken directly by index, which is the IMU notebook's rule. The two are
  identical wherever the window's timestamps are monotonic, and section 1c prints how many windows differ.
- **Inner validation.** The default is `INNER_VAL_RULE = "fallback_2026"`: ~15 % of each class's training missions,
  at least 1, with seed 2026. This is the rule the standalone audio and IMU runs used. v27/v28's own rule
  (`_split_inner_val`) actually used **seed 2027**; set `INNER_VAL_RULE = "v27"` to use it instead.
- **New IMU encoder: the existing projection is already convolutional.** v22/v24/v28's IMU projection is
  `Conv1d(6→64, k5) → BN → GELU → Conv1d(64→256, k3) → BN → GELU`. The one new `Conv1d(6→6, k=7, padding 3)` goes
  in front of it (258 parameters).
  - It keeps 6 channels, so the existing projection is reused byte-for-byte.
  - k = 7 extends the receptive field at the input from 5 to 11 samples (≈ 0.2 s at 128 samples per 2.56 s).
  - There is no nonlinearity between the new conv and the first existing conv, so the new layer acts as a learned
    linear pre-filter. It adds reach, not depth.
- **New audio encoder.** The mask m = sigmoid(w), w ∈ ℝ⁸⁰, is initialized at w = 0 (m = 0.5). It feeds straight into
  a Linear layer, so it is a bounded re-parameterization of that layer's input columns. It mainly works as an
  interpretable feature-importance diagnostic.
- **⚠️ Gates vs pre-norm.** Each Mamba block is `x + Mamba(LayerNorm(x))`. LayerNorm is scale-invariant per token, so
  multiplying a whole modality's tokens by a positive scalar g does **not** change what the first Mamba layer sees. The
  gate acts only through the residual stream, and through it on block 2, the attention pooling and the classifier.
  F1/F2 are built exactly as specified. To check the gates are not functionally inert, section 7 reports a
  **content-adaptivity effect**: how much the predictions change when each window's gates are replaced by their
  test-set mean.

## 2. Setup
Environment and dataset (attached, else `git clone --depth 1`), real `mamba-ssm` (no fallback), every constant, and
seeding. `CUBLAS_WORKSPACE_CONFIG=:4096:8` is set in the **first** line of the first cell, because it only takes
effect before CUDA initializes.

In [1]:
# ==============================================================================
# 2a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"    # cuBLAS reproducibility; only effective before CUDA initializes

import ast, copy, csv, gc, glob, hashlib, json, math, random, re, subprocess, sys, time, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
WORK = Path("/kaggle/working") if IS_KAGGLE else (Path("/content") if IS_COLAB else Path.cwd() / "v29_outputs")
BASE = WORK / "tii"
WORK.mkdir(parents=True, exist_ok=True)
BASE.mkdir(parents=True, exist_ok=True)
REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p):
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_att = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _att += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _att:
    DATA_ROOT = _att[0]
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT = _repo / "Dataset"
assert _is_dataset_root(DATA_ROOT)

CACHE_DIR = WORK / "v29_cache"
PROBS_DIR = WORK / "v29_probs"
CKPT_DIR = WORK / "v29_checkpoints"
RESULTS_CSV = WORK / "gated_fusion_results.csv"
TABLE_CSV = WORK / "v29_results_table.csv"
for _d in (CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"DATA_ROOT {DATA_ROOT} | results {RESULTS_CSV} | probs {PROBS_DIR} | checkpoints {CKPT_DIR}")

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git (--depth 1) ...


Cloning into '/kaggle/working/tii/UAV-Realistic-Fault-Dataset'...


DATA_ROOT /kaggle/working/tii/UAV-Realistic-Fault-Dataset/Dataset | results /kaggle/working/gated_fusion_results.csv | probs /kaggle/working/v29_probs | checkpoints /kaggle/working/v29_checkpoints


Updating files: 100% (217/217), done.


In [2]:
# ==============================================================================
# 2b. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)
# ==============================================================================
import importlib, importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: real mamba-ssm needs a GPU. Enable a GPU accelerator and re-run.")
if importlib.util.find_spec("mamba_ssm") is None:
    _ok = (_pip(["packaging", "ninja"]) and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output). Check internet + GPU, restart, re-run.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"`from mamba_ssm import Mamba` failed: {exc!r}. No fallback is used.") from exc
device = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} ({MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}) | torch "
      f"{torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

  pip install packaging ninja
  pip install causal-conv1d>=1.2.0 --no-build-isolation
  pip install mamba-ssm --no-build-isolation
mamba_ssm 2.3.2.post1 (mamba_ssm.modules.mamba_simple.Mamba) | torch 2.10.0+cu128 | CUDA 12.8 | GPU Tesla T4


In [3]:
# ==============================================================================
# 2c. CONSTANTS — every fixed value lives here
# ==============================================================================
# --- run matrix ---
SEEDS_REQUIRED = [42, 123]
EXTRA_SEED_2024 = False          # OPTIONAL extension: set True and re-run; logged runs are skipped (resumable)
SEEDS = SEEDS_REQUIRED + ([2024] if EXTRA_SEED_2024 else [])
RUN_F2 = True                    # SECONDARY ablation — skippable; runs after F1 within each fold
MODELS = ["A", "I", "D", "F0", "F1"] + (["F2"] if RUN_F2 else [])
FUSED_MODELS = {"D", "F0", "F1", "F2"}
NEW_MODELS = ["F0", "F1"] + (["F2"] if RUN_F2 else [])

# --- missions / days (v23/v24/v27; asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"
STEADY = (0.10, 0.90)

# --- inner validation ---
INNER_VAL_RULE = "fallback_2026"   # "fallback_2026" (standalone audio/IMU runs; spec) or "v27" (v27/v28's rule, seed 2027)
INNER_VAL_FRACTION = 0.15
INNER_VAL_FALLBACK_SEED = 2026
INNER_VAL_SEED_V27 = 2027
assert INNER_VAL_RULE in ("fallback_2026", "v27")

# --- audio features (audio notebook == v28 A1) ---
N_SLOTS, DROP_SLOT, FRAME = 6, 5, 256
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
BUFFER_LEN = N_SLOTS * FRAME                     # 1536
KEEP_BINS = list(range(1, 17))
N_AUDIO_FEAT = len(KEEP_SLOTS) * len(KEEP_BINS)  # 80
AUDIO_STD_EPS = 1e-6

# --- paired windows (v24 CORRECTED_ALIGNMENT) ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
TA_AUDIO = 64
TS_SENSOR = 128
MIN_SENSOR_SAMPLES_PER_WINDOW = 10
N_SENSOR_CHANNELS = 6
SENSOR_STD_EPS = 1e-6

# --- Condition D (v22/v24/v27, unchanged) ---
N_AUDIO_BINS = N_AUDIO_FEAT      # D's audio input width: v24 used 64 mono bins; the shared protocol feeds the 80 A1 features
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                  # used only if the lag bias were ON; Condition D -> beta = 0.0
LAG_BIAS_HIDDEN = 32
LAG_BIAS_PARAMS = 97
LAG_BIAS_LN_EPS = 1e-5
HTT_TIME_EMBED_ENABLED = False
LACA_LAG_BIAS_ENABLED = False
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496
DROPOUT = 0.15
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- new encoders / gates ---
IMU_PRECONV_KERNEL = 7           # one Conv1d, 'same' padding; see the design notes for the choice
IMU_PRECONV_CHANNELS = 6         # keeps the existing projection's input width (6) -> existing projection reused unchanged
GATE_HIDDEN = 64                 # F1/F2 gate MLP: Linear(512 -> 64) -> GELU -> Linear(64 -> 2 or 1)

# --- optimization (v24 == v28) ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- analysis (fixed before any result) ---
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
NOISE_FLOOR_F1 = 0.016           # project's seed-to-seed SD of macro-F1 (v15/v16)
GATE_STD_THRESHOLD = 0.05        # a gate whose across-window SD on the test set is below this is "near-constant"
N_HASH_SAMPLE = 25

# --- determinism investigation ---
DET_FOLD, DET_SEED, DET_EPOCHS = 1, 42, 2

print(f"Models {MODELS} | seeds {SEEDS} (EXTRA_SEED_2024={EXTRA_SEED_2024}) | RUN_F2={RUN_F2} | "
      f"{len(MODELS) * len(DAYS) * len(SEEDS)} runs | inner validation: {INNER_VAL_RULE}")

Models ['A', 'I', 'D', 'F0', 'F1', 'F2'] | seeds [42, 123] (EXTRA_SEED_2024=False) | RUN_F2=True | 60 runs | inner validation: fallback_2026


In [5]:
# ==============================================================================
# 2d. SEEDING, DETERMINISM FLAGS, HOOK HELPER
# ==============================================================================
def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)   # section 5 switches it off and on to test its effect
seed_all(42)

HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=True):
    ok = bool(ok)
    HOOKS[name].append((ok, msg))
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

## 3. Data — parse once, then build ONE set of paired windows
- **3a — parse** each mission once, then cache it:
  - **IMU:** the IMU notebook's schema-first parser, **file order, never sorted**;
  - **audio:** the audio notebook's per-chunk 80 features, plus `recording_started_time`.
- **3b — metadata:** the corrected audio anchor, the exclusion rule, and the mission / class / day counts.
- **3c — paired windows (v24 CORRECTED_ALIGNMENT)**, with the pairing hooks.

In [6]:
# ==============================================================================
# 3a. PARSE (all 99 folders: the 4 exclusions are computed from metadata, not assumed) -> v29_cache/<mission>.npz
# ==============================================================================
def unwrap_message(obj):                                   # IMU notebook, unchanged
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def get_vec(d, candidates):                                # IMU notebook, unchanged
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        vals, ok = [], True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None


def parse_imu_file(path):
    """IMU notebook's parser: rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER."""
    rows, bad = [], 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, ["accelerometer_m_s2", "accelerometer_m_s2[{i}]", "accelerometer", "acceleration"])
                gyro = get_vec(obj, ["gyro_rad", "gyro_rad[{i}]", "gyro", "angular_velocity"])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1
    return np.asarray(rows, dtype=np.float64).reshape(-1, 7), bad


def parse_mission(mdir):
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}
    s_path = sorted(mdir.glob("*SensorCombined*.jsonl"))[0]
    a_path = sorted(mdir.glob("*AudioBuffer*.jsonl"))[0]
    imu, bad = parse_imu_file(s_path)
    chunks, lens, rst, rsts = [], Counter(), None, set()
    with open(a_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            x = rec.get("data")
            if x is None or len(x) == 0:
                continue
            if "recording_started_time" in rec:
                rsts.add(float(rec["recording_started_time"]))
                if rst is None:
                    rst = float(rec["recording_started_time"])          # v24: the FIRST record's value
            lens[len(x)] += 1
            chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal}"
    assert rst is not None, f"{mdir.name}: no recording_started_time — the audio cannot be anchored"
    keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)
    raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)
    n_all = len(chunks)
    del chunks
    frames = raw.reshape(len(raw), FRAME, N_SLOTS)                # sample i -> frame i // 6, slot i % 6
    slot6_zero = bool(np.all(frames[:, :, DROP_SLOT] == 0))
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                      # per-chunk mean removed, per channel
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)    # (n, 5, 129)
    feats = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT).astype(np.float32)   # sqrt|rFFT|, channel-major
    out = {"imu_t": imu[:, 0], "imu_x": imu[:, 1:].astype(np.float32), "imu_bad": bad, "feats": feats,
           "keep_idx": keep_idx, "n_all_chunks": n_all, "rst": rst, "n_rst_values": len(rsts), "slot6_zero": slot6_zero}
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, cache)
    return out


def mission_day(name):
    d = name[8:18].replace("_", "-")
    return DAY_MERGE.get(d, d)


ALL_DIRS, LABEL_OF = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not (class_dir.is_dir() and class_dir.name.isdigit()):
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF[mdir.name] = int(class_dir.name)
_t0 = time.time()
PARSED = {}
for i, mdir in enumerate(ALL_DIRS, 1):
    PARSED[mdir.name] = parse_mission(mdir)
    if i % 10 == 0 or i == len(ALL_DIRS):
        print(f"  parsed {i}/{len(ALL_DIRS)} missions | {time.time() - _t0:.0f} s")

  parsed 10/99 missions | 16 s
  parsed 20/99 missions | 32 s
  parsed 30/99 missions | 50 s
  parsed 40/99 missions | 66 s
  parsed 50/99 missions | 82 s
  parsed 60/99 missions | 98 s
  parsed 70/99 missions | 114 s
  parsed 80/99 missions | 131 s
  parsed 90/99 missions | 148 s
  parsed 99/99 missions | 162 s


In [7]:
# ==============================================================================
# 3b. METADATA — v24's corrected audio anchor, exclusion rule, mission / class / day counts (HOOK data)
# ==============================================================================
def timebase(P):
    """v24 CORRECTED_ALIGNMENT time axes. IMU: FILE ORDER, seconds from the first sample. Audio: anchored at
    recording_started_time; chunk duration = (mission duration - offset) / number of chunks."""
    t_us = P["imu_t"]
    s_times = (t_us - t_us[0]) / 1e6
    duration = float(s_times[-1] - s_times[0])
    offset = (float(P["rst"]) - float(t_us[0])) / 1e6
    chunk = (duration - offset) / int(P["n_all_chunks"])
    a_times = (offset + np.arange(int(P["n_all_chunks"]), dtype=np.float64) * chunk)[P["keep_idx"]]
    return s_times, a_times, duration, offset, chunk


META = []
for name, P in PARSED.items():
    s_times, a_times, dur, off, ch = timebase(P)
    META.append({"mission": name, "class": LABEL_OF[name], "day": mission_day(name), "duration_s": dur, "offset_s": off,
                 "ratio": off / dur, "chunk_ms": 1e3 * ch, "n_imu": len(s_times), "imu_inversions": int((np.diff(s_times) < 0).sum()),
                 "n_chunks": int(P["n_all_chunks"]), "chunks_dropped": int(P["n_all_chunks"]) - len(P["keep_idx"]),
                 "rst_values": int(P["n_rst_values"]), "slot6_zero": bool(P["slot6_zero"])})
META = pd.DataFrame(META).set_index("mission")
EXCLUDED = set(META.index[META.ratio >= EXCLUDE_RATIO])
KEPT = sorted(set(META.index) - EXCLUDED)
M = META.loc[KEPT]
LABEL = {n: int(M.loc[n, "class"]) for n in KEPT}
DAY = {n: M.loc[n, "day"] for n in KEPT}
_cls = np.bincount(M["class"], minlength=N_CLASSES).tolist()
_days = [int((M.day == d).sum()) for d in DAYS]
print(META.loc[sorted(EXCLUDED), ["class", "day", "offset_s", "duration_s", "ratio"]].round(3).to_string())
print(f"\nKept {len(KEPT)} missions | max kept ratio {M.ratio.max():.3f} | median offset {M.offset_s.median():.2f} s | "
      f"chunk duration {M.chunk_ms.median():.2f} ms (median; v24's corrected-anchor formula) | "
      f"non-modal chunks dropped {int(M.chunks_dropped.sum())} | missions with IMU timestamp inversions "
      f"{int((M.imu_inversions > 0).sum())} (total {int(M.imu_inversions.sum())})")
print(pd.crosstab(M.day, M["class"]).reindex(DAYS).to_string())
hook("data", EXCLUDED == EXPECTED_EXCLUDED, f"exclusion set == the 4 known names: {sorted(EXCLUDED)}")
hook("data", len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
     f"{len(KEPT)} missions, classes {_cls}, days {_days} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, {EXPECTED_DAY_COUNTS})")
hook("data", bool(M.slot6_zero.all()) and bool((M.rst_values == 1).all()),
     f"audio slot 6 identically zero in {int(M.slot6_zero.sum())}/{len(M)}; recording_started_time constant in "
     f"{int((M.rst_values == 1).sum())}/{len(M)}")

                             class         day  offset_s  duration_s  ratio
mission                                                                    
rosbag2_2021_12_29-09_10_31      1  2021-12-31   577.157      94.656  6.097
rosbag2_2021_12_31-09_58_13      2  2021-12-31    96.062      85.090  1.129
rosbag2_2021_12_31-10_04_22      2  2021-12-31    99.344     100.703  0.987
rosbag2_2021_12_31-10_12_31      0  2021-12-31    98.431      83.251  1.182

Kept 95 missions | max kept ratio 0.339 | median offset 19.04 s | chunk duration 14.08 ms (median; v24's corrected-anchor formula) | non-modal chunks dropped 0 | missions with IMU timestamp inversions 95 (total 481924)
class       0  1  2  3  4
day                      
2021-12-22  2  2  2  2  2
2021-12-31  7  7  6  8  8
2022-01-04  3  2  4  4  4
2022-01-05  3  4  2  2  4
2022-01-10  3  4  4  4  2
[HOOK data] PASS — exclusion set == the 4 known names: ['rosbag2_2021_12_29-09_10_31', 'rosbag2_2021_12_31-09_58_13', 'rosbag2_2021_12_31-10_0

True

In [8]:
# ==============================================================================
# 3c. ONE SET OF PAIRED WINDOWS (v24 CORRECTED_ALIGNMENT): audio (64 x 80) and IMU (128 x 6, file order) from the SAME
#     real time span of the SAME mission. Every model below reads these arrays (unimodal models ignore one half).
# ==============================================================================
def paired_windows(name, P):
    s_times, a_times, dur, _, _ = timebase(P)
    tel, feats = P["imu_x"], P["feats"]
    max_time = min(s_times[-1], a_times[-1])
    out, i = [], 0
    t_start = max(0.0, float(a_times[0]))                  # v24: CORRECTED_WINDOW_START = "audio_onset"
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue
        s_idx = np.where(s_mask)[0]
        pick = s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]
        tau_s_abs = s_times[pick]
        x_s = tel[pick].astype(np.float32)                 # real file-order samples (IMU notebook rule)
        x_s_v24 = np.stack([np.interp(tau_s_abs, s_times[s_mask], tel[s_mask, c]) for c in range(N_SENSOR_CHANNELS)],
                           axis=1).astype(np.float32)      # v24's np.interp form, kept only for the comparison count
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
            tau_a_abs = a_times[a_pick]
            x_a = feats[a_pick].astype(np.float32)         # == v24's np.interp at exact (strictly increasing) chunk times
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
            x_a = np.stack([np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0) for c in range(N_AUDIO_FEAT)],
                           axis=1).astype(np.float32)
        rel = (t_start + WINDOW_SEC / 2) / dur
        out.append({"window_id": f"{name}#{i:04d}", "mission": name, "cls": LABEL[name], "day": DAY[name],
                    "t_start": t_start, "t_end": t_end,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady"),
                    "x_a": x_a, "x_s": x_s, "tau_a": ((tau_a_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "tau_s": ((tau_s_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "v24_interp_differs": bool(not np.array_equal(x_s, x_s_v24)),
                    "v24_interp_maxdiff": float(np.abs(x_s - x_s_v24).max())})
        i += 1
        t_start += STEP_SEC                                 # v24: accumulated float step
    return out


_W = []
for name in KEPT:
    _W.extend(paired_windows(name, PARSED[name]))
WMETA = pd.DataFrame([{k: v for k, v in w.items() if k not in ("x_a", "x_s", "tau_a", "tau_s")} for w in _W])
XA = np.stack([w["x_a"] for w in _W])                      # (N, 64, 80)
XS = np.stack([w["x_s"] for w in _W])                      # (N, 128, 6)
TAU_A = np.stack([w["tau_a"] for w in _W])                 # (N, 64)   in window units (v24 dataset: tau / window_sec)
TAU_S = np.stack([w["tau_s"] for w in _W])                 # (N, 128)
YW = WMETA.cls.values.astype(np.int64)
del _W
gc.collect()
print(f"Paired windows: {len(WMETA)} | XA {XA.shape} | XS {XS.shape} | per class {np.bincount(YW, minlength=N_CLASSES).tolist()} | "
      f"per phase {WMETA.phase.value_counts().to_dict()}")
print(f"IMU: windows where the file-order direct pick differs from v24's np.interp form: "
      f"{int(WMETA.v24_interp_differs.sum())} / {len(WMETA)} (max abs diff {WMETA.v24_interp_maxdiff.max():.4g}) — "
      "nonzero only where a window's IMU timestamps go backwards")

# ---- pairing hooks ----
_span_ok = bool((TAU_A >= -1e-6).all() and (TAU_A <= 1 + 1e-6).all() and (TAU_S >= -1e-6).all() and (TAU_S <= 1 + 1e-6).all())
hook("pairing", _span_ok and WMETA.window_id.is_unique,
     "every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission "
     "(both come from a single record: same mission id, same span)")
_rng_h = np.random.default_rng(0)
_sample = np.sort(_rng_h.choice(len(WMETA), size=min(N_HASH_SAMPLE, len(WMETA)), replace=False))
_hash_ok = True
for j in _sample:
    r = WMETA.iloc[j]
    rebuilt = {w["window_id"]: w for w in paired_windows(r.mission, PARSED[r.mission])}[r.window_id]
    same = sha16(rebuilt["x_a"], rebuilt["x_s"]) == sha16(XA[j], XS[j]) and abs(rebuilt["t_start"] - r.t_start) < 1e-12
    _hash_ok &= same
hook("pairing", _hash_ok, f"hash check on {len(_sample)} sampled windows: stored (audio, IMU) == rebuilt from the raw "
                          f"mission at the recorded (mission, t_start)")
_fin = bool(np.isfinite(XA).all() and np.isfinite(XS).all())
hook("pairing", _fin and XA.shape[1:] == (TA_AUDIO, N_AUDIO_FEAT) and XS.shape[1:] == (TS_SENSOR, N_SENSOR_CHANNELS),
     f"shapes audio {XA.shape[1:]}, IMU {XS.shape[1:]}; all finite: {_fin}")
PAIRED_SHA = sha16(XA, XS)
print(f"Paired-window content sha1: {PAIRED_SHA}")

Paired windows: 5238 | XA (5238, 64, 80) | XS (5238, 128, 6) | per class [961, 1094, 974, 1118, 1091] | per phase {'steady': 4637, 'end': 601}
IMU: windows where the file-order direct pick differs from v24's np.interp form: 5238 / 5238 (max abs diff 84.07) — nonzero only where a window's IMU timestamps go backwards
[HOOK pairing] PASS — every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission (both come from a single record: same mission id, same span)
[HOOK pairing] PASS — hash check on 25 sampled windows: stored (audio, IMU) == rebuilt from the raw mission at the recorded (mission, t_start)
[HOOK pairing] PASS — shapes audio (64, 80), IMU (128, 6); all finite: True
Paired-window content sha1: 44c04ca18ec7a131


## 4. Folds — 5 day-out folds, inner validation, fold-train normalization and class weights
- **Fold k** tests on every mission of `DAYS[k]` (12-29 merged into 12-31) and trains on the other four days.
- **Inner validation:** ~15 % of each class's training missions, at least 1, following `INNER_VAL_RULE`. Nothing is
  selected on a test fold.
- **Normalization:** per-feature audio z-score (80 features) and per-channel IMU z-score (6 channels), both computed
  from the fold's **training windows only**.
- **Class weights** are inverse class frequency over the fold's training windows (the v24/v28 rule).
- These statistics are shared by all 6 models. A hash of them is printed per fold.

In [9]:
# ==============================================================================
# 4. FOLDS (identical for all 6 models)
# ==============================================================================
def inner_val_fallback_2026(pool, k):
    """Standalone audio/IMU notebooks' rule: per class, ~15% (min 1) of the training missions -> validation, seed 2026."""
    rng = np.random.default_rng([INNER_VAL_FALLBACK_SEED, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if LABEL[m] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


def inner_val_v27(pool, k):
    """v27/v28 `_split_inner_val(pool, protocol_idx=0, fold_idx=k)` (fingerprint 58e3c037c513ace2), seed 2027."""
    rng = np.random.default_rng([INNER_VAL_SEED_V27, 0, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if LABEL[m] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        assert len(ms) - n_val >= 1
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


inner_val = inner_val_fallback_2026 if INNER_VAL_RULE == "fallback_2026" else inner_val_v27
FOLDS = {}
for k, day in enumerate(DAYS):
    test = sorted(m for m in KEPT if DAY[m] == day)
    tr, va = inner_val([m for m in KEPT if DAY[m] != day], k)
    FOLDS[k] = {"day": day, "train": sorted(tr), "val": sorted(va), "test": test}
    assert not (set(tr) & set(va)) and not ((set(tr) | set(va)) & set(test)) and set(tr) | set(va) | set(test) == set(KEPT)
    assert day not in {DAY[m] for m in tr + va}
hook("folds", sorted(m for f in FOLDS.values() for m in f["test"]) == sorted(KEPT),
     "5 day-out test folds partition the 95 missions; no mission in two of train/val/test; test day never in training")

FOLD_PREP = {}
for k, f in FOLDS.items():
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    assert set(WMETA.mission.values[idx["train"]]) == set(f["train"])
    a_tr = XA[idx["train"]].reshape(-1, N_AUDIO_FEAT)
    s_tr = XS[idx["train"]].reshape(-1, N_SENSOR_CHANNELS)
    a_mu, a_sd = a_tr.mean(0, dtype=np.float64).astype(np.float32), (a_tr.std(0, dtype=np.float64) + AUDIO_STD_EPS).astype(np.float32)
    s_mu, s_sd = s_tr.mean(0, dtype=np.float64).astype(np.float32), (s_tr.std(0, dtype=np.float64) + SENSOR_STD_EPS).astype(np.float32)
    del a_tr, s_tr
    counts = np.bincount(YW[idx["train"]], minlength=N_CLASSES)
    FOLD_PREP[k] = {"idx": idx, "a_mu": a_mu, "a_sd": a_sd, "s_mu": s_mu, "s_sd": s_sd, "zhash": sha16(a_mu, a_sd, s_mu, s_sd),
                    "class_weights": (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)}

print(f"Inner validation: {INNER_VAL_RULE}")
print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions/class':<22}{'windows tr/val/test':<22}{'z-score hash':<18}class weights")
for k, f in FOLDS.items():
    P = FOLD_PREP[k]
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train', 'val', 'test')):<22}"
          f"{str(np.bincount([LABEL[m] for m in f['test']], minlength=N_CLASSES).tolist()):<22}"
          f"{'/'.join(str(len(P['idx'][s])) for s in ('train', 'val', 'test')):<22}{P['zhash']:<18}"
          f"{np.round(P['class_weights'], 3).tolist()}" + ("  <- SMALL fold (10 missions)" if f["day"] == SMALL_FOLD_DAY else ""))

[HOOK folds] PASS — 5 day-out test folds partition the 95 missions; no mission in two of train/val/test; test day never in training
Inner validation: fallback_2026
fold test day    missions tr/val/test  test missions/class   windows tr/val/test   z-score hash      class weights
0    2021-12-22  72/13/10              [2, 2, 2, 2, 2]       3932/704/602          b5726f1a0d498224  [1.0440000295639038, 0.9919999837875366, 1.0460000038146973, 0.972000002861023, 0.953000009059906]  <- SMALL fold (10 missions)
1    2021-12-31  49/10/36              [7, 7, 6, 8, 8]       2700/564/1974         720c42d948aed91e  [1.1180000305175781, 0.9520000219345093, 0.9940000176429749, 0.9419999718666077, 1.0110000371932983]
2    2022-01-04  67/11/17              [3, 2, 4, 4, 4]       3680/621/937          dd723497b43897b8  [1.0709999799728394, 0.9330000281333923, 1.1360000371932983, 0.9359999895095825, 0.9559999704360962]
3    2022-01-05  69/11/15              [3, 4, 2, 2, 4]       3852/596/790          5359d

## 5. Models — A, I, D (unchanged), F0, F1, F2
**Condition D**'s classes are copied **verbatim** from `version 24/minor-project-v24.ipynb`, where they are
byte-identical to v22 and v27. The only difference is `build_fusion_blocks`, which here has no fallback branch: real
`mamba_ssm` only.

**A / I** are v28's `UnimodalMamba`:
- the audio path is `Linear(80→256) → LayerNorm → GELU → Linear`;
- the IMU path is `Conv1d(6→64, k5) → BN → GELU → Conv1d(64→256, k3) → BN → GELU`;
- then `LayerNorm(z + modality embedding)` → 2 Mamba blocks → attention pooling → classifier.

**F0 / F1 / F2** share the new encoders and D's HTT-off tokenization, `LayerNorm(z + e_modality)`, with one shared
LayerNorm:
- **Summaries:** `A_summary = mean(A_tokens)` and `I_summary = mean(I_tokens)`. This is **mean pooling**, which is
  what D uses for its per-modality summaries. D's attention pooling is used only on the fused sequence.
- **Losses:** these same summaries feed InfoNCE and, through `aux_a` / `aux_s`, the KL consistency term. They also
  feed the F1/F2 gate MLP. No second pooled representation is computed.
- **F0:** `H = [A_tokens ; I_tokens]`.
- **F1:** `[g_a, g_i] = softmax(MLP([A_summary ; I_summary]))`, where the MLP is `Linear(512→64) → GELU →
  Linear(64→2)`, and `H = [g_a·A_tokens ; g_i·I_tokens]`.
- **F2:** `g_a = σ(f_a([A_summary ; I_summary]))` and `g_i = σ(f_i(·))`, where f_a and f_i are two separate MLPs,
  each `Linear(512→64) → GELU → Linear(64→1)`.
- **Then:** H → 2 Mamba blocks (d_state 16, d_conv 2, expand 2) → attention pooling → classifier.

In [10]:
# ==============================================================================
# 5a. CONDITION D — v24's model + loss classes, VERBATIM (v22 == v24 == v27); build_fusion_blocks: real mamba_ssm only
# ==============================================================================
import math


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    """v24's function without the CPU fallback branch: real mamba_ssm.Mamba blocks only."""
    return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])


# ---- VERBATIM from version 24/minor-project-v24.ipynb (model cell + AeroSyncLoss from the training cell) ----
# sha1[:10] of each source (identical in v22, v24, v27): ContinuousTimeEmbedding 6eb422a2ea, HeterogeneousTemporalTokenization f378aaef09, GridLayerNorm 3a64dba834, PairwiseLACA da33fc4f61, BiModalLACAFusion 567cf3d5d6, MambaBlock b88a5bf138, AeroSyncMamba 9d843e21a5, verify_architecture a7c78390a9, AeroSyncLoss 468046c494

class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}

In [11]:
# ==============================================================================
# 5b. A / I (v28 UnimodalMamba) and the NEW encoders + F0 / F1 / F2
# ==============================================================================
class UnimodalMamba(nn.Module):
    """v28's single-modality path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention pooling
    -> classifier. Reads only its own modality from the shared paired windows."""

    def __init__(self, modality):
        super().__init__()
        self.modality = modality
        if modality == "audio":
            self.audio_encoder = nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
                                               nn.Linear(D_MODEL, D_MODEL))
        else:
            self.sensor_encoder = nn.Sequential(
                nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
                nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, xa, xs, ta, ts, gate_override=None):
        z = self.audio_encoder(xa) if self.modality == "audio" else self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        h = self.token_norm(z + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1)), None, None, {}


class GatedAudioEncoder(nn.Module):
    """NEW audio encoder: ONE learnable elementwise feature mask m = sigmoid(w), w in R^80 (a fixed learned mask, NOT
    input-dependent), then the existing audio projection."""

    def __init__(self):
        super().__init__()
        self.gate_logit = nn.Parameter(torch.zeros(N_AUDIO_FEAT))      # m = 0.5 at init; zeros_ draws no RNG
        self.proj = nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))

    def mask(self):
        return torch.sigmoid(self.gate_logit)

    def forward(self, xa):
        return self.proj(self.mask() * xa)


class ConvIMUEncoder(nn.Module):
    """NEW IMU encoder: ONE Conv1d(6 -> 6, k=7, 'same' padding) before the existing IMU projection (unchanged)."""

    def __init__(self):
        super().__init__()
        self.pre_conv = nn.Conv1d(N_SENSOR_CHANNELS, IMU_PRECONV_CHANNELS, kernel_size=IMU_PRECONV_KERNEL,
                                  padding=IMU_PRECONV_KERNEL // 2)
        self.proj = nn.Sequential(
            nn.Conv1d(IMU_PRECONV_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())

    def forward(self, xs):                                  # (B, T, 6) -> (B, T, D)
        return self.proj(self.pre_conv(xs.transpose(1, 2))).transpose(1, 2)


class GatedFusionMamba(nn.Module):
    """F0: [A ; I] (no gate). F1: softmax gate [g_a, g_i]. F2: two independent sigmoid gates.
    A_summary / I_summary (mean pooling, as D) feed the gate AND the InfoNCE / KL losses (no second pooled vector)."""

    def __init__(self, kind):
        super().__init__()
        assert kind in ("F0", "F1", "F2")
        self.kind = kind
        self.audio_encoder = GatedAudioEncoder()
        self.sensor_encoder = ConvIMUEncoder()
        self.modality_embeddings = nn.Embedding(2, D_MODEL)          # D's HTT with the time embedding OFF
        self.token_norm = nn.LayerNorm(D_MODEL)
        if kind == "F1":
            self.gate_mlp = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 2))
        elif kind == "F2":
            self.gate_a = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 1))
            self.gate_i = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 1))
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))
        self.aux_a = nn.Linear(D_MODEL, N_CLASSES)
        self.aux_s = nn.Linear(D_MODEL, N_CLASSES)

    def forward(self, xa, xs, ta, ts, gate_override=None):
        B = xa.shape[0]
        e = self.modality_embeddings.weight
        A = self.token_norm(self.audio_encoder(xa) + e[0].view(1, 1, -1))     # (B, 64, D)
        I = self.token_norm(self.sensor_encoder(xs) + e[1].view(1, 1, -1))    # (B, 128, D)
        a_sum, i_sum = A.mean(dim=1), I.mean(dim=1)
        s = torch.cat([a_sum, i_sum], dim=-1)
        if self.kind == "F0":
            g_a = g_i = torch.ones(B, device=xa.device)
        elif self.kind == "F1":
            g = F.softmax(self.gate_mlp(s), dim=-1)
            g_a, g_i = g[:, 0], g[:, 1]
        else:
            g_a, g_i = torch.sigmoid(self.gate_a(s)).squeeze(-1), torch.sigmoid(self.gate_i(s)).squeeze(-1)
        if gate_override is not None:                                          # diagnostics only: constant gates
            g_a = torch.full((B,), float(gate_override[0]), device=xa.device)
            g_i = torch.full((B,), float(gate_override[1]), device=xa.device)
        H = torch.cat([g_a.view(B, 1, 1) * A, g_i.view(B, 1, 1) * I], dim=1)   # (B, 192, D)
        for block in self.mamba_blocks:
            H = block(H)
        w = F.softmax(self.pool_attn(H), dim=1)
        logits = self.classifier((H * w).sum(dim=1))
        return (logits, {"z_a": a_sum, "z_s": i_sum}, {"y_a": self.aux_a(a_sum), "y_s": self.aux_s(i_sum)},
                {"g_a": g_a, "g_i": g_i})


class DWrapper(nn.Module):
    """Condition D with the common interface; the wrapped AeroSyncMamba is v24's class, unmodified."""

    def __init__(self):
        super().__init__()
        self.net = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
        self.mamba_blocks = self.net.mamba_blocks

    def forward(self, xa, xs, ta, ts, gate_override=None):
        logits, pooled, aux, lags = self.net(xa, xs, ta, ts)
        return logits, pooled, aux, {}


def build_model(name):
    if name == "A":
        return UnimodalMamba("audio")
    if name == "I":
        return UnimodalMamba("sensor")
    if name == "D":
        return DWrapper()
    return GatedFusionMamba(name)

In [12]:
# ==============================================================================
# 5c. PARAMETER COUNTS, ONE-LINE ARCHITECTURE SUMMARY, identical real-Mamba config (HOOK models)
# ==============================================================================
ARCH = {
    "A": "audio 80 -> MLP(256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE",
    "I": "IMU 6ch -> CNN(k5,k3 -> 256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE",
    "D": "v24 AeroSyncMamba: MLP audio + CNN IMU -> HTT(time OFF) -> LACA x-attn (4 heads, lag x 0) -> [Ha;SEP;Hs] -> 2x Mamba | CE+InfoNCE+KL",
    "F0": "gate(80) audio + Conv1d(k7) IMU -> HTT-off tokens -> [A;I] -> 2x Mamba | CE+InfoNCE+KL",
    "F1": "as F0 + softmax gate MLP(512->64->2) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL",
    "F2": "as F0 + two sigmoid gate MLPs(512->64->1) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL",
}
N_PARAMS, _cfg = {}, {}
seed_all(42)
for name in MODELS:
    m = build_model(name)
    N_PARAMS[name] = sum(p.numel() for p in m.parameters())
    blocks = list(m.mamba_blocks)
    _cfg[name] = (tuple(type(b.mamba) for b in blocks), tuple((b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) for b in blocks),
                  sum(p.numel() for p in m.mamba_blocks.parameters()))
    extra = ""
    if name in NEW_MODELS:
        extra = (f" | added: audio gate {m.audio_encoder.gate_logit.numel()} params, IMU pre-Conv1d "
                 f"{sum(p.numel() for p in m.sensor_encoder.pre_conv.parameters())} params"
                 + (f", gate MLP {sum(p.numel() for p in m.gate_mlp.parameters()):,}" if name == "F1" else "")
                 + (f", gate MLPs {sum(p.numel() for n_, p in m.named_parameters() if n_.startswith(('gate_a', 'gate_i'))):,}" if name == "F2" else ""))
    print(f"  {name:<3} {N_PARAMS[name]:>11,} params | {ARCH[name]}{extra}")
    if name == "D":
        verify_architecture(m.net)
        assert not m.net.htt.time_enabled and m.net.laca.laca_as.beta == 0.0 and m.net.laca.laca_sa.beta == 0.0
        print(f"      D check: HTT time embedding {'ON' if m.net.htt.time_enabled else 'OFF'} | LACA beta "
              f"{m.net.laca.laca_as.beta}/{m.net.laca.laca_sa.beta} | LACA heads {m.net.laca.laca_as.num_heads} (Q/K/V + "
              f"softmax cross-attention ACTIVE) | gamma_as/gamma_sa init {float(m.net.laca.gamma_as):.1f}/{float(m.net.laca.gamma_sa):.1f}")
    del m
_ref = _cfg[MODELS[0]]
hook("models", all(all(t is MAMBA_CLASS for t in c[0]) and c[1] == ((MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND),) * MAMBA_NUM_BLOCKS
                   and c[2] == EXPECTED_MAMBA_PARAMS for c in _cfg.values()) and MAMBA_CLASS.__module__.startswith("mamba_ssm"),
     f"all {len(MODELS)} models use {MAMBA_NUM_BLOCKS} x {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} (d_state "
     f"{MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}, expand {MAMBA_EXPAND}; {EXPECTED_MAMBA_PARAMS:,} Mamba params each)")
torch.cuda.empty_cache()

  A     1,012,358 params | audio 80 -> MLP(256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE
  I       977,350 params | IMU 6ch -> CNN(k5,k3 -> 256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE
  D     1,692,884 params | v24 AeroSyncMamba: MLP audio + CNN IMU -> HTT(time OFF) -> LACA x-attn (4 heads, lag x 0) -> [Ha;SEP;Hs] -> 2x Mamba | CE+InfoNCE+KL
      D check: HTT time embedding OFF | LACA beta 0.0/0.0 | LACA heads 4 (Q/K/V + softmax cross-attention ACTIVE) | gamma_as/gamma_sa init 0.5/0.5
  F0    1,067,554 params | gate(80) audio + Conv1d(k7) IMU -> HTT-off tokens -> [A;I] -> 2x Mamba | CE+InfoNCE+KL | added: audio gate 80 params, IMU pre-Conv1d 258 params
  F1    1,100,516 params | as F0 + softmax gate MLP(512->64->2) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL | added: audio gate 80 params, IMU pre-Conv1d 258 params, gate MLP 32,962
  F2    1,133,348 params | as F0 + two sigmoid gate MLPs(512->64->1) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL | added: audio gate

/tmp/ipykernel_58/804851709.py:32: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"softmax cross-attention ACTIVE) | gamma_as/gamma_sa init {float(m.net.laca.gamma_as):.1f}/{float(m.net.laca.gamma_sa):.1f}")


## 6. Training (shared by all 6 models)
The procedure is v24/v28's:
- AdamW with lr 1e-4 and weight decay 1e-4; cosine annealing to 1e-6 over 20 epochs, stepped per epoch;
- gradient clipping at 1.0; batch 32;
- 20 epochs with no early stopping; the checkpoint kept is the epoch with the best validation macro-F1 (strictly
  greater).

**Loss:**
- **D / F0 / F1 / F2:** v24's `AeroSyncLoss`, verbatim: class-weighted CE + 0.1·InfoNCE(z_a, z_s) + 0.1·KL(aux heads ‖
  fused, fused detached).
- **A / I:** class-weighted CE only.

**Each run:**
- starts with `seed_all(seed)` and builds a fresh model;
- prints an initial-weights fingerprint;
- uses a seeded DataLoader generator with `worker_init_fn`;
- frees GPU memory afterwards.

In [13]:
# ==============================================================================
# 6. METRICS, PREDICTION, train_run
# ==============================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score)
from scipy import stats

LABELS = list(range(N_CLASSES))


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=LABELS))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def fold_arrays(k, split):
    P_, idx = FOLD_PREP[k], FOLD_PREP[k]["idx"][split]
    return (((XA[idx] - P_["a_mu"]) / P_["a_sd"]).astype(np.float32), ((XS[idx] - P_["s_mu"]) / P_["s_sd"]).astype(np.float32),
            TAU_A[idx], TAU_S[idx], YW[idx])


def make_loader(arrs, shuffle, seed):
    ds = TensorDataset(*[torch.from_numpy(np.ascontiguousarray(a)) for a in arrs])
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn, generator=make_generator(seed) if shuffle else None)


@torch.no_grad()
def predict(model, loader, gate_override=None):
    model.eval()
    probs, ga, gi = [], [], []
    for xa, xs, ta, ts, _ in loader:
        logits, _, _, ex = model(xa.to(device), xs.to(device), ta.to(device), ts.to(device), gate_override=gate_override)
        probs.append(F.softmax(logits, dim=-1).float().cpu().numpy())
        if "g_a" in ex:
            ga.append(ex["g_a"].float().cpu().numpy())
            gi.append(ex["g_i"].float().cpu().numpy())
    return np.concatenate(probs), (np.concatenate(ga) if ga else None), (np.concatenate(gi) if gi else None)


def mission_level(missions, y, P):
    df = pd.DataFrame(P, columns=LABELS)
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g["y"].first().values, g[LABELS].mean().values, g["y"].first().index.values


def train_run(name, k, seed, epochs=EPOCHS, save=True, verbose=True, record_steps=False):
    t0 = time.time()
    seed_all(seed)                                           # at the START of every run
    model = build_model(name).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    assert all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks)
    cw = torch.tensor(FOLD_PREP[k]["class_weights"], device=device)
    criterion = AeroSyncLoss(cw) if name in FUSED_MODELS else nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(fold_arrays(k, "train"), True, seed)
    va = make_loader(fold_arrays(k, "val"), False, seed)
    te_arrs = fold_arrays(k, "test")
    te = make_loader(te_arrs, False, seed)
    tag = f"[{name} fold {k} ({FOLDS[k]['day']}) seed {seed}]"
    if verbose:
        print(f"{tag} init {init_sha1} | {N_PARAMS[name]:,} params | windows train {len(tr.dataset)} / val "
              f"{len(va.dataset)} / test {len(te.dataset)} | z-score {FOLD_PREP[k]['zhash']}")
    best_f1, best_state, best_epoch, history, step_losses = 0.0, None, None, [], []
    for epoch in range(1, epochs + 1):
        model.train()
        run = defaultdict(float)
        nb = 0
        for xa, xs, ta, ts, y in tr:
            xa, xs, ta, ts, y = (t.to(device, non_blocking=True) for t in (xa, xs, ta, ts, y))
            optimizer.zero_grad()
            logits, pooled, aux, _ = model(xa, xs, ta, ts)
            if name in FUSED_MODELS:
                loss, parts = criterion(logits, y, pooled, aux)
            else:
                loss, parts = criterion(logits, y), {}
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            run["loss"] += loss.item()
            for kk, v in parts.items():
                run[kk] += v
            nb += 1
            if record_steps:
                step_losses.append(loss.item())
        scheduler.step()
        assert math.isfinite(run["loss"]), f"{tag} non-finite loss"
        vP, _, _ = predict(model, va)
        vf1 = compute_metrics(va.dataset.tensors[4].numpy(), vP)["macro_f1"]
        history.append((run["loss"] / nb, vf1))
        improved = vf1 > best_f1
        if improved:
            best_f1, best_state, best_epoch = vf1, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            extra = f" (cls {run['loss_cls'] / nb:.4f}, align {run['loss_align'] / nb:.4f}, cons {run['loss_cons'] / nb:.4f})" if parts else ""
            print(f"{tag} epoch {epoch:02d}/{epochs} | train loss {run['loss'] / nb:.4f}{extra} | val macro-F1 {vf1:.4f}"
                  + ("  * best" if improved else ""))
    if best_state is None:
        best_state, best_epoch = copy.deepcopy(model.state_dict()), epochs
    model.load_state_dict(best_state)
    res = {"model": name, "fold": k, "fold_day": FOLDS[k]["day"], "seed": seed, "init_sha1": init_sha1, "history": history,
           "step_losses": step_losses, "best_epoch": best_epoch, "best_val_macro_f1": round(best_f1, 6),
           "zstats_sha1": FOLD_PREP[k]["zhash"], "n_params": N_PARAMS[name]}
    if save:
        P, ga, gi = predict(model, te)
        y = te_arrs[4]
        meta = WMETA.iloc[FOLD_PREP[k]["idx"]["test"]]
        my, mP, _ = mission_level(meta.mission.values, y, P)
        res.update({f"win_{kk}": v for kk, v in compute_metrics(y, P).items()})
        res.update({f"mis_{kk}": v for kk, v in compute_metrics(my, mP).items()})
        extra = {}
        if name in ("F1", "F2"):
            # content-adaptivity effect: replace each window's gates by their test-set mean, compare predictions
            P_const, _, _ = predict(model, te, gate_override=(float(ga.mean()), float(gi.mean())))
            res.update({"g_a_mean": float(ga.mean()), "g_a_std": float(ga.std()), "g_a_min": float(ga.min()), "g_a_max": float(ga.max()),
                        "g_i_mean": float(gi.mean()), "g_i_std": float(gi.std()), "g_i_min": float(gi.min()), "g_i_max": float(gi.max()),
                        "both_ambiguous_frac": float(((ga < 0.5) & (gi < 0.5)).mean()),
                        "gate_effect_mean_abs_dp": float(np.abs(P - P_const).mean()),
                        "gate_effect_argmax_changed": float((P.argmax(1) != P_const.argmax(1)).mean())})
            extra.update({"g_a": ga.astype(np.float32), "g_i": gi.astype(np.float32)})
            print(f"{tag} GATES (test): g_a mean {ga.mean():.3f} sd {ga.std():.3f} [{ga.min():.3f}, {ga.max():.3f}] | g_i mean "
                  f"{gi.mean():.3f} sd {gi.std():.3f} [{gi.min():.3f}, {gi.max():.3f}] | "
                  f"{'VARIES' if max(ga.std(), gi.std()) > GATE_STD_THRESHOLD else 'NEAR-CONSTANT'} (sd threshold "
                  f"{GATE_STD_THRESHOLD}) | both<0.5: {res['both_ambiguous_frac']:.3f} | constant-gate effect: mean|dp| "
                  f"{res['gate_effect_mean_abs_dp']:.4f}, argmax changed {res['gate_effect_argmax_changed']:.3f}")
        if name in NEW_MODELS:
            mask = model.audio_encoder.mask().detach().float().cpu().numpy()
            extra["audio_mask"] = mask.astype(np.float32)
            if k == 0 and seed == SEEDS[0]:
                top = np.argsort(-mask)[:10]
                print(f"{tag} learned audio mask (80 = 5 ch x bins 1-16): top 10 " + ", ".join(
                    f"ch{j // 16 + 1}/bin{j % 16 + 1}={mask[j]:.3f}" for j in top) + f" | min {mask.min():.3f} max {mask.max():.3f}")
        fname = f"{name}_{k}_{seed}"
        np.savez(PROBS_DIR / f"{fname}.npz", window_id=meta.window_id.values.astype(str), mission=meta.mission.values.astype(str),
                 day=meta.day.values.astype(str), cls=y, phase=meta.phase.values.astype(str), probs=P.astype(np.float32), **extra)
        torch.save(best_state, CKPT_DIR / f"{fname}.pt")
        res.update({"class_weights": json.dumps(np.round(FOLD_PREP[k]["class_weights"], 4).tolist()),
                    "n_train_windows": len(tr.dataset), "n_val_windows": len(va.dataset), "n_test_windows": len(te.dataset),
                    "probs_file": str(PROBS_DIR / f"{fname}.npz"), "checkpoint": str(CKPT_DIR / f"{fname}.pt")})
    res["train_seconds"] = round(time.time() - t0, 1)
    del model, optimizer, scheduler, best_state, tr, va, te
    gc.collect()
    torch.cuda.empty_cache()
    return res

## 7. Determinism investigation (runs before the sweep)
The IMU-only notebook's gate failed its determinism check: seed 42, 2 epochs, run twice. The initial weights were
identical, and the losses differed only in the 9th significant digit (1.2589503781 vs 1.2589503764). This section
**investigates** that failure on Condition D rather than repeating the check.
- **(a)** D, seed 42, 2 epochs, twice, with `cudnn.deterministic = True`, `cudnn.benchmark = False` and
  `use_deterministic_algorithms` **OFF**.
- **(b)** If (a) fails: the same again with `torch.use_deterministic_algorithms(True, warn_only=True)`.
  `CUBLAS_WORKSPACE_CONFIG=:4096:8` was set at the very top of the notebook, the only point where it takes effect,
  because cuBLAS reads it when CUDA initializes. Every warning PyTorch raises about a non-deterministic op is captured
  and printed.
- **(c)** Component isolation. Each piece gets a fixed input and weights, and a forward + backward pass run twice:
  - a real `MambaBlock`;
  - the IMU CNN encoder;
  - D's LACA fusion;
  - D's HTT tokenization (its `nn.Embedding` backward);
  - the audio MLP.

  Only components whose gradients differ between identical passes can be the source. The size of the run-to-run
  variation (max |Δ loss| over every step) is then compared with the ±0.016 seed-to-seed macro-F1 noise floor.

The sweep then runs with the (b) settings. Kernel-level jitter does **not** stop the notebook: it is a different
concern from an unseeded component.

In [14]:
# ==============================================================================
# 7. DETERMINISM INVESTIGATION on Condition D (fold 1, seed 42, 2 epochs, twice)
# ==============================================================================
def flag_state():
    return (f"cudnn.deterministic={torch.backends.cudnn.deterministic}, cudnn.benchmark={torch.backends.cudnn.benchmark}, "
            f"deterministic_algorithms={torch.are_deterministic_algorithms_enabled()} "
            f"(warn_only={torch.is_deterministic_algorithms_warn_only_enabled()}), "
            f"CUBLAS_WORKSPACE_CONFIG={os.environ.get('CUBLAS_WORKSPACE_CONFIG')!r}")


def det_pair(label):
    with warnings.catch_warnings(record=True) as wl:
        warnings.simplefilter("always")
        r1 = train_run("D", DET_FOLD, DET_SEED, epochs=DET_EPOCHS, save=False, verbose=False, record_steps=True)
        r2 = train_run("D", DET_FOLD, DET_SEED, epochs=DET_EPOCHS, save=False, verbose=False, record_steps=True)
    s1, s2 = np.array(r1["step_losses"]), np.array(r2["step_losses"])
    out = {"init_identical": r1["init_sha1"] == r2["init_sha1"], "steps_identical": bool(np.array_equal(s1, s2)),
           "first_diverging_step": int(np.argmax(s1 != s2)) if not np.array_equal(s1, s2) else None,
           "max_abs_step_loss_diff": float(np.abs(s1 - s2).max()),
           "max_abs_epoch_loss_diff": float(max(abs(a[0] - b[0]) for a, b in zip(r1["history"], r2["history"]))),
           "max_abs_val_f1_diff": float(max(abs(a[1] - b[1]) for a, b in zip(r1["history"], r2["history"]))),
           "nondet_warnings": sorted({str(w.message).split("\n")[0][:160] for w in wl if "determinis" in str(w.message).lower()})}
    print(f"({label}) {flag_state()}")
    for e, (a, b) in enumerate(zip(r1["history"], r2["history"]), 1):
        print(f"    epoch {e}: run 1 loss {a[0]:.10f} val-F1 {a[1]:.6f} | run 2 loss {b[0]:.10f} val-F1 {b[1]:.6f}")
    print(f"    init identical {out['init_identical']} | {len(s1)} steps bit-identical: {out['steps_identical']} | first "
          f"diverging step {out['first_diverging_step']} | max |d step loss| {out['max_abs_step_loss_diff']:.3e} | max |d epoch "
          f"loss| {out['max_abs_epoch_loss_diff']:.3e} | max |d val macro-F1| {out['max_abs_val_f1_diff']:.3e}")
    for w in out["nondet_warnings"]:
        print(f"    PyTorch non-determinism warning: {w}")
    return out


DET = {}
torch.use_deterministic_algorithms(False)
DET["a"] = det_pair("a: cudnn flags only")
if DET["a"]["steps_identical"]:
    print("(a) PASS — bit-identical without use_deterministic_algorithms.")
torch.use_deterministic_algorithms(True, warn_only=True)
DET["b"] = det_pair("b: + use_deterministic_algorithms(True, warn_only=True) + CUBLAS :4096:8")


def grad_repeat(module, make_inputs, n=2):
    """Forward + backward twice on the SAME weights and inputs; max |d grad| over all parameters, and max |d output|."""
    module = module.to(device).train()
    outs, grads = [], []
    for _ in range(n):
        module.zero_grad(set_to_none=True)
        out = make_inputs(module)
        out.float().pow(2).mean().backward()
        outs.append(out.detach().clone())
        grads.append([p.grad.detach().clone() for p in module.parameters() if p.grad is not None])
    dg = max(float((a - b).abs().max()) for a, b in zip(grads[0], grads[1]))
    return float((outs[0] - outs[1]).abs().max()), dg


torch.manual_seed(0)
_B = BATCH_SIZE
_xa = torch.randn(_B, TA_AUDIO, D_MODEL, device=device)
_xs = torch.randn(_B, TS_SENSOR, D_MODEL, device=device)
_ta = torch.rand(_B, TA_AUDIO, device=device)
_ts = torch.rand(_B, TS_SENSOR, device=device)
_seq = torch.randn(_B, TA_AUDIO + 1 + TS_SENSOR, D_MODEL, device=device)
_imu = torch.randn(_B, N_SENSOR_CHANNELS, TS_SENSOR, device=device)
_aud = torch.randn(_B, TA_AUDIO, N_AUDIO_FEAT, device=device)
torch.manual_seed(1)
COMPONENTS = {
    "MambaBlock (mamba_ssm CUDA kernels)": (MambaBlock(D_MODEL, MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND), lambda m: m(_seq)),
    "IMU CNN encoder (cuDNN conv + BN)": (nn.Sequential(nn.Conv1d(6, 64, 5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
                                                        nn.Conv1d(64, D_MODEL, 3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU()),
                                          lambda m: m(_imu)),
    "D LACA fusion (Q/K/V softmax x-attn)": (BiModalLACAFusion(D_MODEL, beta=0.0), lambda m: torch.cat(m(_xa, _xs, _ta, _ts)[:2], 1)),
    "D HTT tokenization (nn.Embedding)": (HeterogeneousTemporalTokenization(D_MODEL, time_enabled=False),
                                          lambda m: m(_xa, _ta, modality_idx=0)),
    "audio MLP encoder (cuBLAS)": (nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
                                                 nn.Linear(D_MODEL, D_MODEL)), lambda m: m(_aud)),
}
print("\n(c) Component isolation — identical weights and inputs, forward + backward twice:")
COMP = {}
for cname, (mod, fn) in COMPONENTS.items():
    do, dg = grad_repeat(mod, fn)
    COMP[cname] = (do, dg)
    print(f"    {cname:<42} max |d output| {do:.3e} | max |d grad| {dg:.3e} -> "
          f"{'bit-identical' if do == 0 and dg == 0 else 'NON-DETERMINISTIC'}")
del COMPONENTS
torch.cuda.empty_cache()

_nd = [c for c, (do, dg) in COMP.items() if do != 0 or dg != 0]
if DET["b"]["steps_identical"]:
    DET_VERDICT = "PASS: with use_deterministic_algorithms + CUBLAS config, Condition D is bit-reproducible here."
elif _nd and all("mamba" in c.lower() for c in _nd):
    DET_VERDICT = (f"FAIL persists with every available flag. Component isolation shows ONLY the mamba_ssm block is "
                   f"non-deterministic (gradients differ between identical passes), so the run-to-run variation comes from "
                   f"mamba_ssm's CUDA kernels (the selective-scan / causal-conv backward, non-deterministic by default), "
                   f"not from an unseeded component of this codebase: initial weights are identical "
                   f"({DET['b']['init_identical']}) and the data order is seeded.")
else:
    DET_VERDICT = (f"FAIL persists; non-deterministic components found: {_nd or 'none in isolation'}. "
                   + ("Includes non-Mamba components — check the PyTorch warnings above." if any("mamba" not in c.lower() for c in _nd)
                      else "Isolated components are bit-identical; the jitter arises only in the full training loop."))
_size = DET["b"]["max_abs_step_loss_diff"]
print(f"\nDETERMINISM VERDICT: {DET_VERDICT}")
print(f"SIZE: max |d loss| over {DET_EPOCHS} epochs = {_size:.3e} (per step), {DET['b']['max_abs_epoch_loss_diff']:.3e} (epoch "
      f"mean); max |d val macro-F1| = {DET['b']['max_abs_val_f1_diff']:.3e}. Against the project's seed-to-seed macro-F1 noise "
      f"floor of ±{NOISE_FLOOR_F1}: " + ("negligible (kernel-level jitter, not a reproducibility bug)."
                                        if DET["b"]["max_abs_val_f1_diff"] < NOISE_FLOOR_F1 / 10 else
                                        "NOT negligible — treat same-seed repeats as noisy."))
hook("determinism", DET["b"]["init_identical"], "identical initial weights for identical seeds (seeding is complete)", stop=False)
hook("determinism", DET["b"]["steps_identical"], "bit-identical loss trajectories (see the verdict above if FAIL)", stop=False)
print(f"Sweep runs with: {flag_state()}")

(a: cudnn flags only) cudnn.deterministic=True, cudnn.benchmark=False, deterministic_algorithms=False (warn_only=False), CUBLAS_WORKSPACE_CONFIG=':4096:8'
    epoch 1: run 1 loss 1.4444160251 val-F1 0.825213 | run 2 loss 1.4444160153 val-F1 0.825213
    epoch 2: run 1 loss 0.7353713649 val-F1 0.948717 | run 2 loss 0.7353713414 val-F1 0.948717
    init identical True | 170 steps bit-identical: False | first diverging step 1 | max |d step loss| 3.576e-07 | max |d epoch loss| 2.349e-08 | max |d val macro-F1| 0.000e+00
(b: + use_deterministic_algorithms(True, warn_only=True) + CUBLAS :4096:8) cudnn.deterministic=True, cudnn.benchmark=False, deterministic_algorithms=True (warn_only=True), CUBLAS_WORKSPACE_CONFIG=':4096:8'
    epoch 1: run 1 loss 1.4444160181 val-F1 0.825213 | run 2 loss 1.4444160230 val-F1 0.825213
    epoch 2: run 1 loss 0.7353713327 val-F1 0.948717 | run 2 loss 0.7353713618 val-F1 0.948717
    init identical True | 170 steps bit-identical: False | first diverging step 4 |

## 8. Sweep — fold-major, then model (A, I, D, F0, F1, F2), then seed; resumable
- **Run order:** a partial session leaves **complete model comparisons** for every fold it finished.
- **Logging:** each run is appended to `gated_fusion_results.csv` immediately. Rows already in the CSV, with their
  probability file present, are skipped. Setting `EXTRA_SEED_2024 = True` later therefore only adds the seed-2024
  runs.
- **F2 order:** F2 for a (fold, seed) runs only after F1 for that (fold, seed) is logged.
- **Per run:** the window probabilities (plus gates and the audio mask for F models) go to
  `v29_probs/{model}_{fold}_{seed}.npz`, and the best checkpoint to `v29_checkpoints/`.

In [15]:
# ==============================================================================
# 8. SWEEP
# ==============================================================================
CSV_FIELDS = ["model", "fold", "fold_day", "seed", "n_train_windows", "n_val_windows", "n_test_windows",
              "win_accuracy", "win_macro_f1", "win_balanced_accuracy", "win_macro_auroc",
              "mis_accuracy", "mis_macro_f1", "mis_balanced_accuracy", "mis_macro_auroc",
              "best_epoch", "best_val_macro_f1", "init_sha1", "zstats_sha1", "class_weights", "n_params", "train_seconds",
              "g_a_mean", "g_a_std", "g_a_min", "g_a_max", "g_i_mean", "g_i_std", "g_i_min", "g_i_max",
              "both_ambiguous_frac", "gate_effect_mean_abs_dp", "gate_effect_argmax_changed",
              "probs_file", "checkpoint", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    return {(r.model, int(r.fold), int(r.seed)): r for r in df.itertuples()
            if (PROBS_DIR / f"{r.model}_{int(r.fold)}_{int(r.seed)}.npz").exists()}


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


PLAN = [(m, k, s) for k in range(len(DAYS)) for m in MODELS for s in SEEDS]
_done = load_done()
print(f"{len(PLAN)} runs planned ({len(MODELS)} models x {len(DAYS)} folds x {len(SEEDS)} seeds) | already logged "
      f"{sum(p in _done for p in PLAN)}")
_t0, _n_new, SWEEP_T0 = time.time(), 0, time.time()
for i, (m, k, s) in enumerate(PLAN, 1):
    _done = load_done()
    if (m, k, s) in _done:
        continue
    if m == "F2" and ("F1", k, s) not in _done:
        print(f"[skip] F2 fold {k} seed {s}: F1 for this fold/seed is not logged yet (F2 runs only after F1)")
        continue
    res = train_run(m, k, s)
    append_row(res)
    _n_new += 1
    n_complete = len([p for p in PLAN if p in load_done()])
    eta = (len(PLAN) - n_complete) * (time.time() - _t0) / _n_new / 60
    print(f"==> {n_complete}/{len(PLAN)} complete | {m} fold {k} seed {s}: window macro-F1 {res['win_macro_f1']:.4f}, "
          f"mission macro-F1 {res['mis_macro_f1']:.4f}, best epoch {res['best_epoch']}, {res['train_seconds']:.0f} s | "
          f"ETA {eta:.0f} min")
SWEEP_MINUTES = (time.time() - SWEEP_T0) / 60
print(f"Sweep finished: {len([p for p in PLAN if p in load_done()])}/{len(PLAN)} runs logged | this session trained "
      f"{_n_new} runs in {SWEEP_MINUTES:.1f} min")

60 runs planned (6 models x 5 folds x 2 seeds) | already logged 0
[A fold 0 (2021-12-22) seed 42] init a2901c1c4c22 | 1,012,358 params | windows train 3932 / val 704 / test 602 | z-score b5726f1a0d498224
[A fold 0 (2021-12-22) seed 42] epoch 01/20 | train loss 0.8839 | val macro-F1 0.7240  * best
[A fold 0 (2021-12-22) seed 42] epoch 02/20 | train loss 0.3671 | val macro-F1 0.8072  * best
[A fold 0 (2021-12-22) seed 42] epoch 03/20 | train loss 0.2305 | val macro-F1 0.8593  * best
[A fold 0 (2021-12-22) seed 42] epoch 04/20 | train loss 0.1968 | val macro-F1 0.8020
[A fold 0 (2021-12-22) seed 42] epoch 05/20 | train loss 0.1658 | val macro-F1 0.8445
[A fold 0 (2021-12-22) seed 42] epoch 06/20 | train loss 0.1261 | val macro-F1 0.8680  * best
[A fold 0 (2021-12-22) seed 42] epoch 07/20 | train loss 0.1101 | val macro-F1 0.8170
[A fold 0 (2021-12-22) seed 42] epoch 08/20 | train loss 0.1477 | val macro-F1 0.8617
[A fold 0 (2021-12-22) seed 42] epoch 09/20 | train loss 0.1040 | val macro-

## 9. Results — the primary table (the SAME protocol for every row)
For every model this section reports the pooled out-of-fold day-out results: each window's probabilities are averaged
over the seeds, and every mission is tested once.

**Levels:**
- **window level;**
- **mission level** (argmax of the mean window probability per mission).

**Columns at each level:**
- accuracy, macro-F1 with its 95 % CI, balanced accuracy, macro-AUROC;
- per-class recall;
- missed faults (faulty → predicted Normal), false alarms (Normal → predicted faulty), and the share of errors that are
  adjacent-class.

The macro-F1 CI comes from a **mission-level cluster bootstrap**: 2000 resamples of missions within class, never
windows.

> **This table supersedes every earlier cross-notebook comparison**: D 0.819 (v27), standalone audio 0.845,
> standalone IMU 0.958. Those used different window constructions. **Only the numbers in this table are directly
> comparable to each other.**

In [16]:
# ==============================================================================
# 9a. POOLED OUT-OF-FOLD TABLE (window + mission level) — read back from v29_probs (works after a resumed session)
# ==============================================================================
def load_npz(m, k, s):
    z = np.load(PROBS_DIR / f"{m}_{k}_{s}.npz")
    return {key: z[key] for key in z.files}


def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap_f1(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Macro-F1 over B resamples of MISSIONS within class (never windows)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    pos = {m: i for i, m in enumerate(ms)}
    cnt, mcls = np.zeros((len(ms), N_CLASSES), int), np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        cms[:, c, :] = cnt[rows[rng.integers(0, len(rows), (B, len(rows)))]].sum(1)
    return np.array([prf_from_cm(cm)[2].mean() for cm in cms])


def level_block(missions, y, P):
    pred = P.argmax(1)
    m = compute_metrics(y, P)
    cm = confusion_matrix(y, pred, labels=LABELS)
    _, rec, _ = prf_from_cm(cm)
    lo, hi = np.percentile(cluster_bootstrap_f1(missions, y, pred), [2.5, 97.5])
    err = pred != y
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "f1_ci_low": lo, "f1_ci_high": hi,
            "balanced_accuracy": m["balanced_accuracy"], "macro_auroc": m["macro_auroc"],
            **{f"recall_c{c}": rec[c] for c in range(N_CLASSES)},
            "faulty_pred_normal": float((pred[y > 0] == 0).mean()), "normal_pred_faulty": float((pred[y == 0] > 0).mean()),
            "adjacent_error_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"), "cm": cm}


_done = load_done()
POOL, SEEDS_USED, FOLD_F1, TABLE = {}, {}, {}, []
for m in MODELS:
    seeds_ok = [s for s in SEEDS if all((m, k, s) in _done for k in range(len(DAYS)))]
    SEEDS_USED[m] = seeds_ok
    if not seeds_ok:
        print(f"{m}: no seed has all 5 folds logged — left out of the table")
        continue
    per_seed = []
    for s in seeds_ok:
        parts = [load_npz(m, k, s) for k in range(len(DAYS))]
        ids = np.concatenate([p["window_id"] for p in parts])
        order = np.argsort(ids)
        per_seed.append({"ids": ids[order], "P": np.concatenate([p["probs"] for p in parts])[order]})
    assert all(np.array_equal(ps["ids"], per_seed[0]["ids"]) for ps in per_seed)
    ids = per_seed[0]["ids"]
    meta = WMETA.set_index("window_id").loc[ids].reset_index()
    P = np.mean([ps["P"] for ps in per_seed], axis=0)
    POOL[m] = {"ids": ids, "meta": meta, "P": P}
    y = meta.cls.values
    wb = level_block(meta.mission.values, y, P)
    my, mP, mnames = mission_level(meta.mission.values, y, P)
    mb = level_block(mnames, my, mP)
    POOL[m].update({"win": wb, "mis": mb})
    for lvl, b in (("window", wb), ("mission", mb)):
        TABLE.append({"model": m, "level": lvl, "seeds": str(seeds_ok), "params": N_PARAMS[m],
                      **{k_: v for k_, v in b.items() if k_ != "cm"}})
    FOLD_F1[m] = {}
    for k, day in enumerate(DAYS):
        sel = (meta.day == day).values
        fmy, fmP, _ = mission_level(meta.mission.values[sel], y[sel], P[sel])
        FOLD_F1[m][k] = {"win": compute_metrics(y[sel], P[sel])["macro_f1"], "mis": compute_metrics(fmy, fmP)["macro_f1"],
                         "win_bal": compute_metrics(y[sel], P[sel])["balanced_accuracy"]}

_ids = [POOL[m]["ids"] for m in POOL]
hook("pairing", all(np.array_equal(a, _ids[0]) for a in _ids) and len(_ids[0]) == len(WMETA),
     f"every model's pooled predictions cover the SAME {len(WMETA)} paired windows (identical window ids across "
     f"{list(POOL)})", stop=False)
TABLE = pd.DataFrame(TABLE)
TABLE.to_csv(TABLE_CSV, index=False)
pd.set_option("display.width", 250)
for lvl in ("window", "mission"):
    t = TABLE[TABLE.level == lvl].set_index("model")
    t["macro-F1 [95% CI]"] = [f"{a:.4f} [{b:.4f}, {c:.4f}]" for a, b, c in zip(t.macro_f1, t.f1_ci_low, t.f1_ci_high)]
    print(f"\n=== POOLED DAY-OUT — {lvl.upper()} LEVEL (seed-averaged; identical protocol for every row) ===")
    print(t[["seeds", "params", "accuracy", "macro-F1 [95% CI]", "balanced_accuracy", "macro_auroc"]
            + [f"recall_c{c}" for c in range(N_CLASSES)]
            + ["faulty_pred_normal", "normal_pred_faulty", "adjacent_error_share"]].round(4).to_string())
print(f"\nsaved: {TABLE_CSV}")
print("\nPer-fold window macro-F1 (seed-averaged):")
print(pd.DataFrame({m: {DAYS[k] + (" (SMALL)" if DAYS[k] == SMALL_FOLD_DAY else ""): v["win"] for k, v in FOLD_F1[m].items()}
                    for m in FOLD_F1}).round(4).to_string())

[HOOK pairing] PASS — every model's pooled predictions cover the SAME 5238 paired windows (identical window ids across ['A', 'I', 'D', 'F0', 'F1', 'F2'])

=== POOLED DAY-OUT — WINDOW LEVEL (seed-averaged; identical protocol for every row) ===
           seeds   params  accuracy        macro-F1 [95% CI]  balanced_accuracy  macro_auroc  recall_c0  recall_c1  recall_c2  recall_c3  recall_c4  faulty_pred_normal  normal_pred_faulty  adjacent_error_share
model                                                                                                                                                                                                            
A      [42, 123]  1012358    0.7929  0.7954 [0.7379, 0.8503]             0.7947       0.9442      1.000     0.8483     0.6304     0.7004     0.7947              0.0007               0.000                0.9926
I      [42, 123]   977350    0.8942  0.8931 [0.8651, 0.9171]             0.8945       0.9871      0.999     0.9004     0.7782  

In [17]:
# ==============================================================================
# 9b. GATE DIAGNOSTICS PER FOLD (F1, F2) + learned audio mask (F0/F1/F2)
# ==============================================================================
_csv = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
GATE_TAB = {}
for m in [x for x in ("F1", "F2") if x in MODELS]:
    rows = _csv[_csv.model == m]
    if rows.empty:
        continue
    cols = ["fold", "fold_day", "seed", "g_a_mean", "g_a_std", "g_a_min", "g_a_max", "g_i_mean", "g_i_std", "g_i_min", "g_i_max",
            "gate_effect_mean_abs_dp", "gate_effect_argmax_changed"] + (["both_ambiguous_frac"] if m == "F2" else [])
    t = rows[cols].sort_values(["fold", "seed"]).copy()
    t["varies (sd > thr)"] = np.maximum(t.g_a_std, t.g_i_std) > GATE_STD_THRESHOLD
    GATE_TAB[m] = t
    print(f"\n=== {m} gate distribution on each fold's TEST windows (per seed) ===")
    print(t.round(4).to_string(index=False))
    # per-class mean gate (pooled over seeds) from the saved per-window gates
    rows_c = []
    for k in range(len(DAYS)):
        for s in SEEDS_USED.get(m, []):
            z = load_npz(m, k, s)
            if "g_a" in z:
                rows_c.append(pd.DataFrame({"fold": k, "cls": z["cls"], "g_a": z["g_a"], "g_i": z["g_i"]}))
    if rows_c:
        gc_ = pd.concat(rows_c)
        print(f"{m} mean gate by TRUE class (all folds/seeds):")
        print(gc_.groupby("cls")[["g_a", "g_i"]].agg(["mean", "std"]).round(3).to_string())
        if m == "F2":
            q = pd.crosstab(np.where(gc_.g_a < 0.5, "g_a<0.5", "g_a>=0.5"), np.where(gc_.g_i < 0.5, "g_i<0.5", "g_i>=0.5"),
                            normalize=True)
            print("F2 joint distribution of (g_a, g_i) over all test windows (fractions):")
            print(q.round(4).to_string())
            print(f"F2 'both ambiguous' (g_a < 0.5 AND g_i < 0.5) per fold: " + ", ".join(
                f"fold {k} {((g.g_a < 0.5) & (g.g_i < 0.5)).mean():.3f}" for k, g in gc_.groupby("fold")))

masks = {m: [load_npz(m, k, s)["audio_mask"] for k in range(len(DAYS)) for s in SEEDS_USED.get(m, [])]
         for m in NEW_MODELS if SEEDS_USED.get(m)}
FEAT_NAMES = [f"ch{c + 1}/bin{b}" for c in range(len(KEEP_SLOTS)) for b in KEEP_BINS]
for m, ms in masks.items():
    mm = np.mean(ms, axis=0)
    top = np.argsort(-mm)[:10]
    print(f"\n{m} learned audio mask sigmoid(w), mean over {len(ms)} runs: top 10 " + ", ".join(
        f"{FEAT_NAMES[j]} {mm[j]:.3f}" for j in top) + f" | range [{mm.min():.3f}, {mm.max():.3f}] | per-channel mean "
        + ", ".join(f"ch{c + 1} {mm.reshape(len(KEEP_SLOTS), -1)[c].mean():.3f}" for c in range(len(KEEP_SLOTS))))


=== F1 gate distribution on each fold's TEST windows (per seed) ===
 fold   fold_day  seed  g_a_mean  g_a_std  g_a_min  g_a_max  g_i_mean  g_i_std  g_i_min  g_i_max  gate_effect_mean_abs_dp  gate_effect_argmax_changed  varies (sd > thr)
    0 2021-12-22    42    0.3070   0.3101   0.0069   0.9273    0.6930   0.3101   0.0727   0.9931                   0.0494                      0.1246               True
    0 2021-12-22   123    0.1604   0.1690   0.0087   0.6560    0.8396   0.1690   0.3440   0.9913                   0.0538                      0.1196               True
    1 2021-12-31    42    0.9618   0.0206   0.9043   0.9765    0.0382   0.0206   0.0235   0.0957                   0.0002                      0.0010              False
    1 2021-12-31   123    0.3226   0.2747   0.0103   0.8793    0.6774   0.2747   0.1207   0.9897                   0.0854                      0.2133               True
    2 2022-01-04    42    0.0525   0.0235   0.0157   0.1268    0.9475   0.0235   0.873

In [18]:
# ==============================================================================
# 9c. PAIRED COMPARISONS ACROSS THE 5 FOLDS (window macro-F1, seed-averaged): mean, SD, wins/5, paired t-test
# ==============================================================================
def paired(a, b, level="win"):
    if a not in FOLD_F1 or b not in FOLD_F1:
        return None
    d = np.array([FOLD_F1[a][k][level] - FOLD_F1[b][k][level] for k in range(len(DAYS))])
    t, p = stats.ttest_rel([FOLD_F1[a][k][level] for k in range(len(DAYS))], [FOLD_F1[b][k][level] for k in range(len(DAYS))])
    return {"comparison": f"{a} − {b}", "level": level, "per-fold diffs": np.round(d, 4).tolist(), "mean": d.mean(),
            "sd": d.std(ddof=1), "wins/5": int((d > 0).sum()), "t": t, "p (paired t, df 4)": p}


COMPARISONS = [("F0", "D", "new encoders + NO cross-attention vs old encoders + content-only cross-attention — "
                           "CONFLATES two changes (encoders AND fusion); does not isolate the encoders"),
               ("F1", "F0", "does the content gate help beyond plain concatenation? (cleanest comparison: same encoders)"),
               ("F2", "F1", "does removing the sum-to-one constraint help?")]
PAIRED = []
for a, b, why in COMPARISONS:
    for lvl in ("win", "mis"):
        r = paired(a, b, lvl)
        if r:
            r["meaning"] = why if lvl == "win" else ""
            PAIRED.append(r)
# best new model vs the better single modality (chosen by pooled window macro-F1 — descriptive: chosen on test data)
_singles = [m for m in ("A", "I") if m in POOL]
_news = [m for m in NEW_MODELS if m in POOL]
BEST_SINGLE = max(_singles, key=lambda m: POOL[m]["win"]["macro_f1"]) if _singles else None
BEST_NEW = max(_news, key=lambda m: POOL[m]["win"]["macro_f1"]) if _news else None
if BEST_SINGLE and BEST_NEW:
    for lvl in ("win", "mis"):
        r = paired(BEST_NEW, BEST_SINGLE, lvl)
        r["meaning"] = (f"best new model ({BEST_NEW}) vs the better single modality ({BEST_SINGLE}); both picked by pooled "
                        f"macro-F1 on the test folds, so this is descriptive") if lvl == "win" else ""
        PAIRED.append(r)
    _perfold_max = {k: max(FOLD_F1[m][k]["win"] for m in _singles) for k in range(len(DAYS))}
    _d = np.array([FOLD_F1[BEST_NEW][k]["win"] - _perfold_max[k] for k in range(len(DAYS))])
    print(f"Stricter check: {BEST_NEW} vs the per-fold max(A, I) (an oracle choice of modality per day): mean "
          f"{_d.mean():+.4f}, wins {int((_d > 0).sum())}/5")
PAIRED_DF = pd.DataFrame(PAIRED)
print(PAIRED_DF.round(4).to_string(index=False))

if "F2" in FOLD_F1 and "F1" in FOLD_F1 and "F2" in GATE_TAB:
    _amb = GATE_TAB["F2"].groupby("fold").both_ambiguous_frac.mean()
    _dd = np.array([FOLD_F1["F2"][k]["win"] - FOLD_F1["F1"][k]["win"] for k in range(len(DAYS))])
    rho, p = stats.spearmanr(_amb.values, _dd)
    print(f"\nF2 − F1 per-fold gain vs F2's per-fold 'both ambiguous' fraction: Spearman rho {rho:+.2f} (p {p:.2f}, n = 5 "
          f"folds — very weak evidence either way) | fractions {np.round(_amb.values, 3).tolist()} | gains {np.round(_dd, 4).tolist()}")

Stricter check: F1 vs the per-fold max(A, I) (an oracle choice of modality per day): mean -0.0110, wins 2/5
comparison level                               per-fold diffs    mean     sd  wins/5       t  p (paired t, df 4)                                                                                                                                                       meaning
    F0 − D   win [-0.0356, -0.0721, -0.0054, 0.0019, -0.0075] -0.0237 0.0306       1 -1.7351              0.1577 new encoders + NO cross-attention vs old encoders + content-only cross-attention — CONFLATES two changes (encoders AND fusion); does not isolate the encoders
    F0 − D   mis                [0.0, -0.0502, 0.0, 0.0, 0.0] -0.0100 0.0224       0 -1.0000              0.3739                                                                                                                                                              
   F1 − F0   win      [0.0034, 0.056, 0.0362, 0.0223, 0.0039]  0.0243 0.0224   

/tmp/ipykernel_58/1750381684.py:45: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(_amb.values, _dd)


## 10. Summary, verdict and verification hooks
**Verdict rule** (fixed before any result):
- A difference counts as real only if the mean per-fold Δ macro-F1 exceeds the project's noise floor (0.016) **and**
  the better model wins at least 4 of 5 folds.
- Otherwise the verdict is "no reliable difference".

With n = 5 folds the paired t-test is reported but is weak.

In [19]:
# ==============================================================================
# 10. SUMMARY (generated from the numbers) + hook table
# ==============================================================================
def verdict(a, b):
    r = paired(a, b, "win")
    if r is None:
        return f"{a} vs {b}: not available (runs missing)"
    if r["mean"] > NOISE_FLOOR_F1 and r["wins/5"] >= 4:
        v = f"{a} BEATS {b}"
    elif r["mean"] < -NOISE_FLOOR_F1 and r["wins/5"] <= 1:
        v = f"{a} is WORSE than {b}"
    else:
        v = f"no reliable difference between {a} and {b}"
    return (f"{v} (mean Δ window macro-F1 {r['mean']:+.4f}, SD {r['sd']:.4f}, wins {r['wins/5']}/5, paired t p = "
            f"{r['p (paired t, df 4)']:.3f})")


_csv = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
lines = ["WHAT v29 TELLS US", ""]
lines.append(f"Protocol: {len(KEPT)} missions, {len(WMETA)} paired windows (v24 CORRECTED_ALIGNMENT; audio 64x80 + IMU "
             f"128x6 file order, same span), 5 day-out folds, inner validation {INNER_VAL_RULE}, seeds per model "
             f"{SEEDS_USED}. Runs logged: {len(_csv)} of {len(PLAN)}; total logged training time "
             f"{_csv.train_seconds.sum() / 60 if len(_csv) else 0:.0f} min.")
lines.append("")
lines.append("Pooled day-out macro-F1 [95% mission-bootstrap CI] — window | mission:")
for m in POOL:
    w, mi = POOL[m]["win"], POOL[m]["mis"]
    lines.append(f"  {m:<3} {w['macro_f1']:.3f} [{w['f1_ci_low']:.3f}, {w['f1_ci_high']:.3f}] | {mi['macro_f1']:.3f} "
                 f"[{mi['f1_ci_low']:.3f}, {mi['f1_ci_high']:.3f}]   ({N_PARAMS[m]:,} params)")
lines += ["", "Paired comparisons (5 folds):",
          "  " + verdict("F1", "F0") + "   <- does content-adaptive gating beat plain concatenation?",
          "  " + verdict("F0", "D") + "   <- CAUTION: conflates new encoders AND removal of LACA cross-attention",
          ("  " + verdict("F2", "F1")) if "F2" in POOL else "  F2 vs F1: F2 not run",
          ("  " + verdict(BEST_NEW, BEST_SINGLE) + f"   <- best new model vs the better single modality") if BEST_NEW and BEST_SINGLE else ""]
if "F1" in GATE_TAB:
    t = GATE_TAB["F1"]
    lines.append(f"F1 gates: mean g_a {t.g_a_mean.mean():.3f}, across-window SD {t.g_a_std.mean():.3f} (threshold {GATE_STD_THRESHOLD}); "
                 f"{int(t['varies (sd > thr)'].sum())}/{len(t)} runs vary; replacing per-window gates by their mean changes "
                 f"{t.gate_effect_argmax_changed.mean():.3f} of predictions (mean |dp| {t.gate_effect_mean_abs_dp.mean():.4f}).")
if "F2" in GATE_TAB:
    lines.append(f"F2: 'both ambiguous' (g_a<0.5 and g_i<0.5) fraction {GATE_TAB['F2'].both_ambiguous_frac.mean():.3f} of test windows.")
lines += ["", f"Determinism: {DET_VERDICT} Size: max |d val macro-F1| {DET['b']['max_abs_val_f1_diff']:.2e} vs noise floor {NOISE_FLOOR_F1}.",
          "", "Caveats:",
          "- D vs F0 is NOT a pure encoder comparison: it changes the encoders AND removes LACA's content-only cross-attention "
          "(and D's SEP token). Accepted simplification given time constraints, not an oversight.",
          "- D here takes the 80 de-interleaved audio features (z-scored), not v24's 64 mono bins, so it is not v27's 0.819.",
          "- Pre-norm Mamba blocks make each token's Mamba input scale-invariant, so the gates act through the residual "
          "stream only; see the constant-gate effect above for how much they actually change predictions.",
          "- This table supersedes the earlier cross-notebook numbers (D 0.819, audio 0.845, IMU 0.958).",
          "- 95 missions, 5 folds, 2 seeds: per-fold paired tests are weak (df = 4)."]
SUMMARY = "\n".join(l for l in lines if l is not None)
print(SUMMARY)
(WORK / "v29_summary.txt").write_text(SUMMARY, encoding="utf-8")

print("\n" + "=" * 100 + "\nVERIFICATION HOOKS\n" + "=" * 100)
for name, res in HOOKS.items():
    print(f"  {name:<12} {'PASS' if all(ok for ok, _ in res) else 'FAIL'}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else 'FAIL'}: {msg}")

WHAT v29 TELLS US

Protocol: 95 missions, 5238 paired windows (v24 CORRECTED_ALIGNMENT; audio 64x80 + IMU 128x6 file order, same span), 5 day-out folds, inner validation fallback_2026, seeds per model {'A': [42, 123], 'I': [42, 123], 'D': [42, 123], 'F0': [42, 123], 'F1': [42, 123], 'F2': [42, 123]}. Runs logged: 60 of 60; total logged training time 74 min.

Pooled day-out macro-F1 [95% mission-bootstrap CI] — window | mission:
  A   0.795 [0.738, 0.850] | 0.864 [0.793, 0.927]   (1,012,358 params)
  I   0.893 [0.865, 0.917] | 0.968 [0.934, 1.000]   (977,350 params)
  D   0.871 [0.823, 0.917] | 0.893 [0.832, 0.948]   (1,692,884 params)
  F0  0.838 [0.784, 0.890] | 0.873 [0.807, 0.937]   (1,067,554 params)
  F1  0.869 [0.821, 0.913] | 0.894 [0.832, 0.948]   (1,100,516 params)
  F2  0.853 [0.802, 0.901] | 0.883 [0.817, 0.946]   (1,133,348 params)

Paired comparisons (5 folds):
  F1 BEATS F0 (mean Δ window macro-F1 +0.0243, SD 0.0224, wins 5/5, paired t p = 0.072)   <- does content-adaptiv